# Chapter 21: Advanced Techniques: Dynamic Matrices, github-script, ChatOps (notebook edition)

## Learning Objectives

- Compute a matrix at run time and validate the plan
- Explain why a false guard turns a failure into a green skipped job
- Authorize ChatOps commands by author_association

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Load the evidence

Every number comes from a real run saved in `fixtures/`.

In [ ]:
import json
S = json.loads((ROOT / "fixtures" / "advanced_ch21_summary.json").read_text())
dm = S["dynamic_matrix"]
print(dm["unguarded"]["valid"]["jobs"])

## 2. The plan contract

A plan must emit a non-empty `include` list; anything else should fail in the plan job.

In [ ]:
def validate_plan(matrix_json):
    """Return the legs of a plan or raise."""
    legs = json.loads(matrix_json)["include"]
    if not isinstance(legs, list) or not legs:
        raise ValueError("matrix must contain at least one leg")
    return legs


legs = validate_plan(dm["plan_report_valid"])
assert [f"run ({x['name']}, {x['python']})" for x in legs] == dm["unguarded"]["valid"]["jobs"][1:]
for bad in ("this is not json", '{"include":[]}'):
    try:
        validate_plan(bad)
        raise AssertionError(bad)
    except (ValueError, KeyError):
        pass
print(len(legs), "legs")

## 3. The guard hides the failure

In [ ]:
def guarded_outcome(count):
    """What `if: count > 0` does to the matrix job."""
    return "runs" if count > 0 else "skipped"


g = dm["guarded_by_count"]
assert guarded_outcome(3) == "runs" and guarded_outcome(0) == "skipped"
assert g["invalid"]["run_job"] == "skipped" and g["invalid"]["conclusion"] == "success"
assert g["invalid"]["count_output"] == ""   # empty, not -1
assert dm["fromjson_empty"]["step_level"]["conclusion"] == "failure"
assert dm["fromjson_empty"]["job_level_if"]["conclusion"] == "skipped"
print("an invalid plan is a green, skipped job")

## 4. ChatOps authorization

In [ ]:
ALLOWED = {"OWNER", "MEMBER", "COLLABORATOR"}


def handle(body, association):
    """Return (outcome, reply) for a comment."""
    _, sub, *rest = body.split()
    if association not in ALLOWED:
        return "denied", f"Not authorized ({association})."
    if sub == "ping":
        return "ping", "pong"
    if sub == "echo":
        return "echo", " ".join(rest) or "(nothing to echo)"
    return "help", "Commands: /tally ping, /tally echo <text>"


co = S["chatops"]
assert handle("/tally echo hello    from   chatops", "OWNER") == ("echo", co["echo_output"])
assert handle("/tally ping", "CONTRIBUTOR")[0] == "denied"
assert len(co["runs"]) == 5 and co["bot_replies_started_runs"] == 0
print("5 comments, 5 runs, bot replies started 0")

## Chapter Link

Read: `learning_modules/chapter_21_advanced_techniques.md`